# Colab setup: VPR project

Run top to bottom. Use **Runtime → Change runtime type → GPU** for SALAD.

Before the first run, someone with the files puts these in a shared Google Drive folder (default name `CS566_VPR`):
- `nordland_full_images.zip`, all 30,352 images (763 MB), or `nordland_smoke_images.zip` for a quick test

The repo is private, so cloning needs a GitHub personal access token with read access to it
(GitHub → Settings → Developer settings → Fine-grained tokens). The token is typed into a hidden prompt and not saved.

In [ ]:
# ---- settings ----
DRIVE_DIR = "/content/drive/MyDrive/CS566_VPR"    # shared folder holding the zip(s); if it was shared with you, add a shortcut to My Drive first
ZIP_NAME  = "nordland_full_images.zip"            # or "nordland_smoke_images.zip"
BRANCH    = "main"                                # or your role branch, e.g. "salad-baseline"
REPO      = "Gradi3nt9/Visual-Place-Recognition-Across-Environmental-Changes"
PERSIST_CACHE = True                              # keep features/descriptors on Drive between sessions

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import getpass, os, subprocess
if not os.path.exists("/content/repo"):
    token = getpass.getpass("GitHub token: ")
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, f"https://{token}@github.com/{REPO}.git", "/content/repo"], check=True)
    # drop the token from the saved remote URL
    subprocess.run(["git", "-C", "/content/repo", "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
    del token
%cd /content/repo
!git log --oneline -3

In [ ]:
# Colab already ships torch, torchvision, opencv, numpy, pillow, pyyaml, tqdm, matplotlib, pytest.
# Don't install requirements.txt here: its torch pin would replace Colab's CUDA build.
!pip -q install pytorch-lightning==2.1.2
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime to GPU)")

In [ ]:
# Unzip onto Colab's local disk (much faster to read than Drive).
import os
os.makedirs("data/nordland", exist_ok=True)
if not os.path.exists("data/nordland/ref"):
    !cp "{DRIVE_DIR}/{ZIP_NAME}" /content/
    !unzip -q "/content/{ZIP_NAME}" -d data/nordland
!ls data/nordland/ref | wc -l; ls data/nordland/query | wc -l

In [ ]:
# Where caches live. On Drive they survive the session (SALAD database descriptors are worth keeping).
CACHE = f"{DRIVE_DIR}/cache" if PERSIST_CACHE else "cache"
os.makedirs(CACHE, exist_ok=True)
SET = f"--set cache_dir={CACHE}"
print(SET)

In [ ]:
# Sanity checks. The full check only passes with the full zip; with the smoke zip it reports missing images, which is expected.
!python -m pytest -q
!python scripts/check_nordland.py --sample 500

In [ ]:
# Pipeline check: random baseline on the smoke split (expect R@1 around 1%)
!python scripts/run_retrieval.py --method random --split nordland_smoke {SET}
!python scripts/evaluate.py results/rankings/nordland_smoke__random.json

## Running real methods
Once `sift_ransac` / `salad` are implemented on your branch:
```
!python scripts/run_retrieval.py --method salad --split nordland_smoke {SET}
!python scripts/run_retrieval.py --method salad --split nordland_dev1  {SET}
!python scripts/evaluate.py results/rankings/nordland_dev1__salad.json
```
`results/` is on Colab's local disk and disappears with the session. Copy anything you need first:
`!cp -r results "{DRIVE_DIR}/results_$(date +%m%d_%H%M)"`